# Module 4.1: NVIDIA Headless Driver Installation, GPU Power Capping & NVIDIA Container Toolkit

## Homelab Expansion (Chapter 2) - Step 4: GPU Acceleration & Workload Migration

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand how Linux exposes an NVIDIA GPU (`GTX 1660 Ti` / `RTX 2060` / `RTX 2070`) on a 2019 ROG Zephyrus to Docker containers via the **NVIDIA Container Toolkit (`nvidia-ctk`)**.
2. Install the official Ubuntu Server headless NVIDIA driver and verify GPU idle power states (`P8` ~4W idle) using `nvidia-smi` and `nvtop`.
3. Configure a persistent `systemd` GPU Power Limit service (`nvidia-smi -pl`) to protect the battery-less laptop's AC power brick from micro-transient current spikes during heavy AI inference.
4. Install upstream **Docker Engine (`docker-ce`)** and **Docker Compose** alongside `nvidia-container-toolkit`.
5. Validate hardware CUDA passthrough inside an isolated container (`docker run --gpus all`).

## 4.1.1 Conceptual Background: Exposing Laptop NVIDIA GPUs to Linux Containers

By default, Docker containers are isolated from host PCIe hardware devices via Linux `cgroups` and namespaces. Even if your host OS has an NVIDIA GeForce GTX 1660 Ti or RTX 20-series GPU, a standard container only sees the CPU.

```mermaid
flowchart TB
    subgraph Container ["Docker Container (immich-machine-learning:release-cuda)"]
        ONNX["ONNX Runtime / PyTorch<br>(CLIP & InsightFace Facial Rec)"]
        CUDALib["CUDA User-Space Libraries (cuDNN / TensorRT)"]
    end
    
    subgraph Runtime ["Host Container Runtime (homelab-gpu)"]
        Docker["Docker Engine + containerd"]
        CTK["NVIDIA Container Toolkit (nvidia-ctk hook)<br>Mounts /dev/nvidia* & Driver .so Files"]
    end
    
    subgraph Kernel ["Ubuntu Server 24.04 LTS Kernel"]
        Driver["NVIDIA Kernel Modules<br>(nvidia.ko, nvidia-uvm.ko, nvidia-modeset.ko)"]
        GPU["2019 ROG Zephyrus NVIDIA GPU<br>(1536–2304 CUDA Cores + NVENC Silicon)"]
    end
    
    ONNX --> CUDALib
    CUDALib --> CTK
    Docker --> CTK
    CTK --> Driver
    Driver --> GPU
```

### Protecting a Battery-Less Laptop with `nvidia-smi` Persistence & Power Capping
Recall from Module 1.1 that gaming laptops with the internal battery removed rely 100% on the external AC adapter (180W/230W). On a 2019 ROG Zephyrus, the NVIDIA GPU has a default TDP between `60W` (Max-Q) and `115W` (Full-Power GX502).
- By enabling **NVIDIA Persistence Mode (`nvidia-smi -pm 1`)**, the Linux kernel keeps the GPU initialized so it drops immediately into its **P8 Idle Performance State (~3W–5W)** when no containers are actively indexing photos, and wakes in `<5ms` when a job arrives.
- By setting a conservative **GPU Power Cap (`nvidia-smi -pl 55` or `-pl 65`)**, you retain **~92% of peak CUDA inference throughput** while cutting peak GPU power spikes by 25%–35%—guaranteeing the AC adapter never trips Over-Current Protection (OCP) and keeping GPU load temperatures below **68°C**!

## 4.1.2 Step-by-Step Laboratory Guide

### Task 4.1.1: Install Headless NVIDIA Server Drivers & `nvtop`
Connect to `homelab-gpu` (`ssh homelab-gpu`) and use Ubuntu's hardware auto-detection tool to install the recommended proprietary NVIDIA headless driver:
```bash
# 1. Inspect detected NVIDIA GPU hardware and recommended driver packages
sudo apt update
sudo apt install -y ubuntu-drivers-common nvtop
ubuntu-drivers devices

# 2. Automatically install the recommended NVIDIA driver (or nvidia-driver-550-server)
sudo ubuntu-drivers autoinstall

# 3. Reboot homelab-gpu to load the nvidia.ko kernel module
sudo reboot
```
Wait 20 seconds and reconnect via `ssh homelab-gpu`, then verify the driver is loaded:
```bash
nvidia-smi
```
You should see the standard `NVIDIA-SMI` table listing your GeForce GTX 1660 Ti / RTX 2060 / 2070, driver version, CUDA version (`12.x`), temperature, and `P8` power state!

### Task 4.1.2: Create a Persistent Battery-Less GPU Power-Cap & Persistence Service
Check the minimum, default, and maximum supported power limits on your specific Zephyrus GPU:
```bash
nvidia-smi -q -d POWER | grep -E "Power Limit|Default Power Limit|Min Power Limit|Max Power Limit"
```
Now create a small `systemd` service that enables **Persistence Mode** and caps peak GPU draw at a safe, cool **60W** (if your `Min Power Limit` is higher than 60W, substitute that value) on every boot:
```bash
sudo tee /etc/systemd/system/nvidia-homelab-powercap.service > /dev/null << 'EOT'
[Unit]
Description=NVIDIA Homelab Persistence Mode and Battery-Less Power Cap
After=syslog.target systemd-modules-load.service

[Service]
Type=oneshot
ExecStart=/usr/bin/nvidia-smi -pm 1
# Set safe 60W power limit (adjust if your GPU Min Power Limit differs)
ExecStart=-/usr/bin/nvidia-smi -pl 60
RemainAfterExit=yes

[Install]
WantedBy=multi-user.target
EOT

sudo systemctl daemon-reload
sudo systemctl enable --now nvidia-homelab-powercap.service
```

### Task 4.1.3: Install Upstream Docker Engine (`docker-ce`) & NVIDIA Container Toolkit
Just as we did on the Raspberry Pi in Chapter 1, install official upstream Docker Engine and then register the **NVIDIA Container Toolkit** runtime:

```bash
# 1. Install Docker Engine from Docker's official apt repository
sudo apt update
sudo apt install -y ca-certificates curl gnupg
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "$VERSION_CODENAME") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null

sudo apt update
sudo apt install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin

# 2. Add user 'pi' to the docker group
sudo usermod -aG docker pi
```

Now install the **NVIDIA Container Toolkit (`nvidia-ctk`)** and configure Docker to use it:
```bash
# 3. Add NVIDIA Container Toolkit GPG key and repository
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg \
  && curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | \
    sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
    sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list

sudo apt update
sudo apt install -y nvidia-container-toolkit

# 4. Register the nvidia runtime hook in /etc/docker/daemon.json and restart Docker
sudo nvidia-ctk runtime configure --runtime=docker
sudo systemctl restart docker
```

### Task 4.1.4: Test NVIDIA GPU Passthrough Inside a Docker Container
Log out and log back into `homelab-gpu` (or run `newgrp docker`) so your group membership takes effect, then launch an ephemeral Ubuntu CUDA container and execute `nvidia-smi` **inside the container**:
```bash
docker run --rm --gpus all nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi
```
When the container prints your ROG Zephyrus NVIDIA GPU table with `0%` utilization and your `60W` power cap active, your GPU container cluster node is officially ready for production workloads!

### 🖼️ Task 4.1.4 Verification: Containerized `nvidia-smi` GPU Passthrough

**Verification Objective:** Capture a screenshot showing the output of `docker run --rm --gpus all nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi` running on `homelab-gpu`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Headless NVIDIA driver installed and verified via `nvidia-smi` on `homelab-gpu`.
- [ ] `nvidia-homelab-powercap.service` active (`Persistence-M: On` and safe power cap configured).
- [ ] Upstream `docker-ce`, `docker-compose-plugin`, and `nvidia-container-toolkit` installed.
- [ ] `docker run --rm --gpus all ... nvidia-smi` succeeds without privilege errors.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 4.2: Offloading Immich Machine Learning & Hardware Transcoding](./04_2_immich_ml_and_transcoding_offload.ipynb).